In [1]:
import numpy as np
import matplotlib.pyplot as plt
from nnfs.datasets import spiral_data

np.set_printoptions(precision=2)

In [64]:
class DenseLayer:
    def __init__(self, dim_in, dim_out):
        # here we are creating a transposed metrix to avoid transpose at each evaluation of a layer
        # therfore here columns represent the neurons and rows represent the nth input wights of each
        # neuron
        self.weights = np.random.randn(dim_in, dim_out)
        self.dweights = np.random.randn(dim_in, dim_out)
        # using zero as bias for sometime
        self.biases = np.zeros((1, dim_out))
        self.dbiases = np.zeros((1, dim_out))        
    
    def forward(self, inputs):
        self.output = np.dot(inputs, self.weights) + self.bias
        return self.output

class ReLU():
    def forward(self, x):
        self.output = np.maximum(0, x)
        return self.output
    
class Softmax():
    def forward(self, x):
        ex = np.exp(x - x.max(axis=1, keepdims=1))
        self.output = ex/ex.sum(axis=1, keepdims=1)
        return self.output
    
class CategoricalCrossEntrophyLoss():
    def forward(self, y_val, y_true):
        
        if len(y_true.shape) == 1:
            likelihood = y_val[np.arange(len(y_val)), np.array(y_true)]
        elif len(y_true.shape) == 2: # using one hot encoding
            likelihood = np.sum(y_val * y_true, axis=1) # elementwise multiplicaiton

        log_likelihood = np.log(np.clip(likelihood, 1e-7, 1-1e-7))
        neg_log_likelihood = -1 * log_likelihood
        return neg_log_likelihood


class Loss():
    def calculate(self, lossess):
        return np.average(lossess)
    
class OptimizerWithDecayAndMomentum: 
    def __init__(self, learning_rate=1.0, decay=0.0, momentum=0.0):
        self.learning_rate = learning_rate
        self.decay = decay
        self.current_learning_rate = learning_rate
        self.iterations = 0
        self.momentum = momentum
        
    def pre_update(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate / (1.0 + self.decay * self.iterations)
    
    def update_params(self, layer):

        if self.momentum:
            if not hasattr(layer, "weight_momentums"):
                layer.weight_momentums = np.zeros_like(layer.weights)
                layer.bias_momentums = np.zeros_like(layer.biases)
            
            weights_updates = self.momentum * layer.weight_momentums - self.current_learning_rate * layer.dweights
            layer.weight_momentums = weights_updates
            bias_updates = self.momentum * layer.bias_momentums - self.current_learning_rate * layer.dbiases
            layer.bias_momentums = bias_updates
        
        else:
            weights_updates = -self.current_learning_rate * layer.dweights
            bias_updates = -self.current_learning_rate * layer.dbiases
        
        layer.weights += weights_updates
        layer.biases +=  bias_updates
    
    def post_update(self):
        # increment iteration count
        self.iterations += 1
    

In [60]:
l = DenseLayer(3, 2)
o = OptimizerWithDecayAndMomentum(learning_rate=1.0, decay=0.1, momentum=0.1)

In [61]:
o.pre_update()
o.update_params(l)
o.post_update()

In [56]:
l.weights

array([[ 0.01, -0.13],
       [-1.58,  1.11],
       [ 0.65, -0.39]])

In [68]:
class OptimizerAdagrad: 
    def __init__(self, learning_rate=1.0, decay=0.0):
        self.learning_rate = learning_rate
        self.decay = decay
        self.current_learning_rate = learning_rate
        self.iterations = 0
        self.epsilon = 1e-7
        
    def pre_update(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate / (1.0 + self.decay * self.iterations)
    
    def update_params(self, layer):
        
        if not hasattr(layer, "weight_cache"):
                layer.weight_cache = np.zeros_like(layer.weights)
                layer.bias_cache = np.zeros_like(layer.biases)
        
        layer.weight_cache += layer.dweights**2
        layer.bias_cache += layer.dbiases**2
        weights_updates = (-self.current_learning_rate * layer.dweights) / (np.sqrt(layer.weight_cache) + self.epsilon)
        bias_updates = (-self.current_learning_rate * layer.dbiases) / (np.sqrt(layer.bias_cache) + self.epsilon)
        
        layer.weights += weights_updates
        layer.biases +=  bias_updates
    
    def post_update(self):
        # increment iteration count
        self.iterations += 1
    

In [70]:
class OptimizerRMSProp: 
    def __init__(self, learning_rate=1.0, decay=0.0, rho=0.9):
        self.learning_rate = learning_rate
        self.decay = decay
        self.current_learning_rate = learning_rate
        self.iterations = 0
        self.epsilon = 1e-7
        
    def pre_update(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate / (1.0 + self.decay * self.iterations)
    
    def update_params(self, layer):
        
        if not hasattr(layer, "weight_cache"):
                layer.weight_cache = np.zeros_like(layer.weights)
                layer.bias_cache = np.zeros_like(layer.biases)
        
        layer.weight_cache = rho*layer.weight_cache + (1-rho)*layer.dweights**2
        layer.bias_cache = rho*layer.bias_cache + (1-rho)*layer.dbiases**2
        weights_updates = (-self.current_learning_rate * layer.dweights) / (np.sqrt(layer.weight_cache) + self.epsilon)
        bias_updates = (-self.current_learning_rate * layer.dbiases) / (np.sqrt(layer.bias_cache) + self.epsilon)
        
        layer.weights += weights_updates
        layer.biases +=  bias_updates
    
    def post_update(self):
        # increment iteration count
        self.iterations += 1
    

In [75]:
# beta1 - momentum factor
# beta2 - cache memory decay rate
class AdamOptimizer:
    def __init__(self, learning_rate=1.0, lr_decay=0.1, beta1=0.9, beta2=0.999):
        self.current_learning_rate = learning_rate
        self.lr_decay = lr_decay
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = 1e-7
        self.iterations = 0
    
    def pre_update(self):
        self.current_learning_rate = self.current_learning_rate / (1 + self.lr_decay * (self.iterations + 1))
    
    def post_update(self):
        self.iterations += 1
        
    def update_weights(self, layer):
        
        momentum_term_weights = beta1 * layer.momentum_weights + (1-beta1) * layer.dweights
        momentum_term_bias = beta1 * layer.momentum_biases + (1-beta1) * layer.dbiases
        
        # at lower iterations the (1-beta1**(self.iterations + 1)) term will be < 1 therefore avoiding cold start 
        # problems due to numeration too low in the beginning. Over time the term becomes ~ 1 and there won't be any 
        # effect in the higher iterations.
        corrected_momentum_term_weights = momentum_term_weights / (1 - beta1**(self.iterations + 1))
        corrected_momentum_term_bias = momentum_term_bias / (1-beta1**(self.iterations + 1))
        
        layer.cache_weights = beta2 * layer.cache_weights + (1-beta2) * (layer.dweights**2)
        layer.cache_bias = beta2 * layer.cache_biases + (1-beta2) * (layer.dbiases**2)
        
        corrected_cache_term_weights = layer.cache_weights / (1-beta2**(self.iterations + 1))
        corrected_cache_term_biases = layer.cache_biases / (1-beta2**(self.iteration + 1))
        
        layer.weights += -self.current_learning_rate * corrected_momentum_term_weights / np.sqrt(corrected_cache_term_weights) + self.epsilon
        layer.biases += -self.current_learning_rate * corrected_momentum_term_bias / np.sqrt(corrected_cache_term_biases) + self.epsilon
        